In [8]:
"""
================================================================================
BUILD HETEROGENEOUS WORD-DOCUMENT GRAPH (BertGCN Style) — NEPALI SAFE VERSION
================================================================================
• Document nodes + Word nodes
• Word-word edges using NPMI (dynamic sliding window)
• Document-word edges using TF-IDF
• Uses BERT embeddings for documents
• Word node features initialized as zero vectors
• Saves adjacency, features, labels, masks
================================================================================
"""

import numpy as np
import scipy.sparse as sp
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from collections import defaultdict
from tqdm import tqdm
import json
import math
import os

# ============================================================================ #
# CONFIGURATION
# ============================================================================ #

class GraphConfig:
    DATA_PATH = 'preprocessed_news02.csv'
    LABELS_PATH = 'labels.npy'
    TRAIN_MASK_PATH = 'train_mask.npy'
    VAL_MASK_PATH = 'val_mask.npy'
    TEST_MASK_PATH = 'test_mask.npy'
    BERT_DOC_FEATURES_PATH = 'node_features.npy'  # (n_docs, 768)

    # Graph hyperparameters (Nepali safe)
    WINDOW_SIZE = 15
    MIN_WORD_FREQ = 3
    MIN_PAIR_COUNT = 3
    NPMI_THRESHOLD = 0.2
    MAX_VOCAB_SIZE = 20000

    # Outputs
    ADJ_OUTPUT_PATH = 'heterogeneous_adj.npz'
    NODE_FEATURES_PATH = 'heterogeneous_features.npy'
    LABELS_OUTPUT_PATH = 'heterogeneous_labels.npy'
    TRAIN_MASK_OUTPUT = 'heterogeneous_train_mask.npy'
    VAL_MASK_OUTPUT = 'heterogeneous_val_mask.npy'
    TEST_MASK_OUTPUT = 'heterogeneous_test_mask.npy'
    NODE_MAPPING_PATH = 'node_mapping.json'
    GRAPH_STATS_PATH = 'heterogeneous_stats.json'

config = GraphConfig()

# ============================================================================ #
# STEP 1: LOAD DATA
# ============================================================================ #

df = pd.read_csv(config.DATA_PATH)
labels = np.load(config.LABELS_PATH)
train_mask = np.load(config.TRAIN_MASK_PATH)
val_mask = np.load(config.VAL_MASK_PATH)
test_mask = np.load(config.TEST_MASK_PATH)

print(f"Documents loaded: {len(df):,}")
print(f"Train: {train_mask.sum():,}, Val: {val_mask.sum():,}, Test: {test_mask.sum():,}")

# ============================================================================ #
# STEP 2: BUILD VOCABULARY
# ============================================================================ #

vectorizer = CountVectorizer(
    max_features=config.MAX_VOCAB_SIZE,
    min_df=config.MIN_WORD_FREQ,
    token_pattern=r'\S+',
    lowercase=False
)

doc_word_freq = vectorizer.fit_transform(df['content'].astype(str).values)
vocab = vectorizer.get_feature_names_out()
vocab_size = len(vocab)
word2id = {word: idx for idx, word in enumerate(vocab)}

print(f"Vocabulary size: {vocab_size:,}")

# ============================================================================ #
# STEP 3: SLIDING WINDOW PMI (NEPALI SAFE)
# ============================================================================ #

word_window_freq = defaultdict(int)
word_pair_freq = defaultdict(int)
windows_total = 0

tokenizer = vectorizer.build_tokenizer()

for doc in tqdm(df['content'].astype(str).values, desc="Sliding windows"):
    words = [w for w in tokenizer(doc) if w in word2id]

    if len(words) < 2:
        continue

    window_size = min(config.WINDOW_SIZE, len(words))

    for i in range(len(words) - window_size + 1):
        window = words[i:i + window_size]
        window_unique = set(window)

        for w in window_unique:
            word_window_freq[w] += 1

        window_list = list(window_unique)
        for j in range(len(window_list)):
            for k in range(j + 1, len(window_list)):
                pair = tuple(sorted((window_list[j], window_list[k])))
                word_pair_freq[pair] += 1

        windows_total += 1

print(f"Total windows: {windows_total:,}")

# ============================================================================ #
# STEP 4: COMPUTE NPMI EDGES
# ============================================================================ #

pmi_edges = []

for (w1, w2), pair_count in tqdm(word_pair_freq.items(), desc="Computing NPMI"):
    if pair_count < config.MIN_PAIR_COUNT:
        continue

    p_w1 = word_window_freq[w1] / windows_total
    p_w2 = word_window_freq[w2] / windows_total
    p_pair = pair_count / windows_total

    pmi = math.log((p_pair + 1e-12) / (p_w1 * p_w2 + 1e-12))
    npmi = pmi / (-math.log(p_pair + 1e-12))

    if npmi > config.NPMI_THRESHOLD:
        pmi_edges.append((word2id[w1], word2id[w2], npmi))

print(f"Word-word edges kept: {len(pmi_edges):,}")

# ============================================================================ #
# STEP 5: DOCUMENT-WORD TF-IDF EDGES
# ============================================================================ #

tfidf_transformer = TfidfTransformer()
doc_word_tfidf = tfidf_transformer.fit_transform(doc_word_freq)

# ============================================================================ #
# STEP 6: BUILD HETEROGENEOUS ADJACENCY
# ============================================================================ #

n_docs = len(df)
n_words = vocab_size
n_nodes = n_docs + n_words

rows = []
cols = []
weights = []

# Word-word edges
for w1_idx, w2_idx, weight in pmi_edges:
    i = n_docs + w1_idx
    j = n_docs + w2_idx
    rows.extend([i, j])
    cols.extend([j, i])
    weights.extend([weight, weight])

# Document-word edges
doc_word_coo = doc_word_tfidf.tocoo()
for doc_idx, word_idx, weight in zip(doc_word_coo.row, doc_word_coo.col, doc_word_coo.data):
    i = doc_idx
    j = n_docs + word_idx
    rows.extend([i, j])
    cols.extend([j, i])
    weights.extend([weight, weight])

adj = sp.coo_matrix((weights, (rows, cols)), shape=(n_nodes, n_nodes), dtype=np.float32)
adj = adj + sp.eye(n_nodes, dtype=np.float32)

# Symmetric normalization: D^(-1/2) A D^(-1/2)
rowsum = np.array(adj.sum(1)).flatten()
d_inv_sqrt = np.power(rowsum, -0.5)
d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
D_inv_sqrt = sp.diags(d_inv_sqrt)
adj = D_inv_sqrt @ adj @ D_inv_sqrt

adj = adj.tocsr()

print(f"Adjacency shape: {adj.shape}")
print(f"Total edges (including self loops): {adj.nnz:,}")

# ============================================================================ #
# STEP 7: NODE FEATURES
# ============================================================================ #

doc_features = np.load(config.BERT_DOC_FEATURES_PATH)  # (n_docs, 768)
embedding_dim = doc_features.shape[1]

word_features = np.zeros((n_words, embedding_dim), dtype=np.float32)
node_features = np.vstack([doc_features, word_features])

print(f"Node feature shape: {node_features.shape}")

# ============================================================================ #
# STEP 8: LABELS AND MASKS
# ============================================================================ #

hetero_labels = np.zeros(n_nodes, dtype=np.int64)
hetero_labels[:n_docs] = labels

hetero_train_mask = np.zeros(n_nodes, dtype=bool)
hetero_val_mask = np.zeros(n_nodes, dtype=bool)
hetero_test_mask = np.zeros(n_nodes, dtype=bool)

hetero_train_mask[:n_docs] = train_mask
hetero_val_mask[:n_docs] = val_mask
hetero_test_mask[:n_docs] = test_mask

# ============================================================================ #
# STEP 9: SAVE EVERYTHING
# ============================================================================ #

sp.save_npz(config.ADJ_OUTPUT_PATH, adj)
np.save(config.NODE_FEATURES_PATH, node_features.astype(np.float32))
np.save(config.LABELS_OUTPUT_PATH, hetero_labels)
np.save(config.TRAIN_MASK_OUTPUT, hetero_train_mask)
np.save(config.VAL_MASK_OUTPUT, hetero_val_mask)
np.save(config.TEST_MASK_OUTPUT, hetero_test_mask)

mapping = {
    'n_docs': n_docs,
    'n_words': n_words,
    'n_nodes': n_nodes,
    'doc_offset': 0,
    'word_offset': n_docs,
    'vocab_size': vocab_size
}

with open(config.NODE_MAPPING_PATH, 'w') as f:
    json.dump(mapping, f, indent=2)

stats = {
    'graph_type': 'Heterogeneous',
    'n_docs': n_docs,
    'n_words': n_words,
    'n_nodes': n_nodes,
    'n_edges': int(adj.nnz),
    'embedding_dim': embedding_dim,
    'window_size': config.WINDOW_SIZE,
    'npmi_threshold': config.NPMI_THRESHOLD,
    'min_pair_count': config.MIN_PAIR_COUNT
}

with open(config.GRAPH_STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)

print("\n✅ Heterogeneous graph successfully built (Nepali-safe version)")


Documents loaded: 122,817
Train: 12,281, Val: 6,141, Test: 104,395
Vocabulary size: 20,000


Sliding windows: 100%|█████████████████████████████████████████████████████████| 122817/122817 [31:43<00:00, 64.54it/s]


Total windows: 25,677,727


Computing NPMI: 100%|██████████████████████████████████████████████████| 35719143/35719143 [00:37<00:00, 946344.80it/s]


Word-word edges kept: 2,476,469


MemoryError: Unable to allocate 269. MiB for an array with shape (35310216,) and data type float64

In [11]:
# Rebuild pmi_edges from existing data
pmi_edges = []

for (w1, w2), pair_count in tqdm(word_pair_freq.items(), desc="Rebuilding NPMI edges"):
    if pair_count < config.MIN_PAIR_COUNT:
        continue

    p_w1 = word_window_freq[w1] / windows_total
    p_w2 = word_window_freq[w2] / windows_total
    p_pair = pair_count / windows_total

    pmi = math.log((p_pair + 1e-12) / (p_w1 * p_w2 + 1e-12))
    npmi = pmi / (-math.log(p_pair + 1e-12))

    if npmi > config.NPMI_THRESHOLD:
        pmi_edges.append((word2id[w1], word2id[w2], npmi))

print(f"Word-word edges rebuilt: {len(pmi_edges):,}")

# Now continue with the memory-efficient adjacency building...

Rebuilding NPMI edges: 100%|███████████████████████████████████████████| 35719143/35719143 [00:52<00:00, 685256.52it/s]

Word-word edges rebuilt: 2,476,469


In [12]:
# ============================================================================ #
# STEP 5: DOCUMENT-WORD TF-IDF EDGES (CHUNKED VERSION)
# ============================================================================ #

tfidf_transformer = TfidfTransformer()
doc_word_tfidf = tfidf_transformer.fit_transform(doc_word_freq)

# ============================================================================ #
# STEP 6: BUILD HETEROGENEOUS ADJACENCY (ULTRA MEMORY-EFFICIENT)
# ============================================================================ #

n_docs = len(df)
n_words = vocab_size
n_nodes = n_docs + n_words

# Estimate edge count more conservatively
estimated_edges = len(pmi_edges) * 2 + doc_word_tfidf.nnz * 2
print(f"Estimated total edges: {estimated_edges:,}")

# Build adjacency in CSR format directly (more memory efficient)
from scipy.sparse import lil_matrix

print("Initializing sparse matrix...")
adj = lil_matrix((n_nodes, n_nodes), dtype=np.float32)

# Add word-word edges
print("Adding word-word edges...")
for w1_idx, w2_idx, weight in tqdm(pmi_edges, desc="Word-word edges"):
    i = n_docs + w1_idx
    j = n_docs + w2_idx
    adj[i, j] = weight
    adj[j, i] = weight

# Free memory
del pmi_edges

# Add document-word edges in chunks (process in CSR format to avoid conversion)
print("Adding doc-word edges in chunks...")
chunk_size = 1000  # Process 1000 documents at a time

doc_word_tfidf_csr = doc_word_tfidf.tocsr()  # Already in CSR, no conversion needed

for start_idx in tqdm(range(0, n_docs, chunk_size), desc="Doc chunks"):
    end_idx = min(start_idx + chunk_size, n_docs)
    chunk = doc_word_tfidf_csr[start_idx:end_idx]
    
    # Process this chunk
    for local_doc_idx in range(chunk.shape[0]):
        global_doc_idx = start_idx + local_doc_idx
        row = chunk.getrow(local_doc_idx)
        
        for word_idx, weight in zip(row.indices, row.data):
            j = n_docs + word_idx
            adj[global_doc_idx, j] = weight
            adj[j, global_doc_idx] = weight

# Free more memory
del doc_word_tfidf, doc_word_tfidf_csr, doc_word_freq

print("Converting to CSR format...")
adj = adj.tocsr()

# Add self-loops
print("Adding self-loops...")
adj = adj + sp.eye(n_nodes, dtype=np.float32)

# Symmetric normalization: D^(-1/2) A D^(-1/2)
print("Normalizing adjacency matrix...")
rowsum = np.array(adj.sum(1)).flatten()
d_inv_sqrt = np.power(rowsum, -0.5)
d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
D_inv_sqrt = sp.diags(d_inv_sqrt)
adj = D_inv_sqrt @ adj @ D_inv_sqrt

adj = adj.tocsr()

print(f"Adjacency shape: {adj.shape}")
print(f"Total edges (including self loops): {adj.nnz:,}")

# Continue with the rest of your code (STEP 7 onwards)...

Estimated total edges: 35,310,216
Initializing sparse matrix...
Adding word-word edges...


Word-word edges: 100%|███████████████████████████████████████████████████| 2476469/2476469 [00:13<00:00, 177580.14it/s]


Adding doc-word edges in chunks...


Doc chunks: 100%|████████████████████████████████████████████████████████████████████| 123/123 [01:40<00:00,  1.22it/s]


Converting to CSR format...
Adding self-loops...
Normalizing adjacency matrix...
Adjacency shape: (142817, 142817)
Total edges (including self loops): 35,453,033


In [15]:
# Check what variables still exist in your current session
print("Checking available variables...")
print(f"word_pair_freq exists: {'word_pair_freq' in dir()}")
print(f"word_window_freq exists: {'word_window_freq' in dir()}")
print(f"windows_total exists: {'windows_total' in dir()}")
print(f"word2id exists: {'word2id' in dir()}")
print(f"vocab exists: {'vocab' in dir()}")
print(f"vocab_size exists: {'vocab_size' in dir()}")
print(f"doc_word_freq exists: {'doc_word_freq' in dir()}")
print(f"doc_word_tfidf exists: {'doc_word_tfidf' in dir()}")
print(f"df exists: {'df' in dir()}")
print(f"vectorizer exists: {'vectorizer' in dir()}")

Checking available variables...
word_pair_freq exists: True
word_window_freq exists: True
windows_total exists: True
word2id exists: True
vocab exists: True
vocab_size exists: True
doc_word_freq exists: False
doc_word_tfidf exists: False
df exists: True
vectorizer exists: True


In [16]:
# ============================================================================ #
# RECREATE MISSING VARIABLES (FAST - you already have the vectorizer!)
# ============================================================================ #

print("Recreating doc_word_freq and doc_word_tfidf...")
doc_word_freq = vectorizer.transform(df['content'].astype(str).values)
print(f"✓ doc_word_freq shape: {doc_word_freq.shape}")

from sklearn.feature_extraction.text import TfidfTransformer
tfidf_transformer = TfidfTransformer()
doc_word_tfidf = tfidf_transformer.fit_transform(doc_word_freq)
print(f"✓ doc_word_tfidf shape: {doc_word_tfidf.shape}")

# ============================================================================ #
# REBUILD PMI_EDGES FROM EXISTING DATA
# ============================================================================ #

pmi_edges = []

for (w1, w2), pair_count in tqdm(word_pair_freq.items(), desc="Rebuilding NPMI edges"):
    if pair_count < config.MIN_PAIR_COUNT:
        continue

    p_w1 = word_window_freq[w1] / windows_total
    p_w2 = word_window_freq[w2] / windows_total
    p_pair = pair_count / windows_total

    pmi = math.log((p_pair + 1e-12) / (p_w1 * p_w2 + 1e-12))
    npmi = pmi / (-math.log(p_pair + 1e-12))

    if npmi > config.NPMI_THRESHOLD:
        pmi_edges.append((word2id[w1], word2id[w2], npmi))

print(f"✓ Word-word edges rebuilt: {len(pmi_edges):,}")

# ============================================================================ #
# BUILD ADJACENCY (MEMORY-EFFICIENT)
# ============================================================================ #

n_docs = len(df)
n_words = vocab_size
n_nodes = n_docs + n_words

# Pre-allocate arrays
total_edges = len(pmi_edges) * 2 + doc_word_tfidf.nnz * 2
print(f"Building adjacency with {total_edges:,} edges...")

rows = np.empty(total_edges, dtype=np.int32)
cols = np.empty(total_edges, dtype=np.int32)
weights = np.empty(total_edges, dtype=np.float32)

idx = 0

# Word-word edges
print("Adding word-word edges...")
for w1_idx, w2_idx, weight in tqdm(pmi_edges, desc="Word-word edges"):
    i = n_docs + w1_idx
    j = n_docs + w2_idx
    rows[idx] = i
    cols[idx] = j
    weights[idx] = weight
    idx += 1
    rows[idx] = j
    cols[idx] = i
    weights[idx] = weight
    idx += 1

del pmi_edges  # Free memory
print(f"✓ Added word-word edges, current idx: {idx:,}")

# Document-word edges
print("Adding doc-word edges...")
doc_word_coo = doc_word_tfidf.tocoo()
for doc_idx, word_idx, weight in tqdm(zip(doc_word_coo.row, doc_word_coo.col, doc_word_coo.data),
                                       total=doc_word_coo.nnz, desc="Doc-word edges"):
    i = doc_idx
    j = n_docs + word_idx
    rows[idx] = i
    cols[idx] = j
    weights[idx] = weight
    idx += 1
    rows[idx] = j
    cols[idx] = i
    weights[idx] = weight
    idx += 1

print(f"✓ Total edges added: {idx:,}")
print("Building sparse matrix...")
adj = sp.coo_matrix((weights, (rows, cols)), shape=(n_nodes, n_nodes), dtype=np.float32)
adj = adj.tocsr()

print(f"✓ Adjacency shape: {adj.shape}, edges: {adj.nnz:,}")

# ============================================================================ #
# NODE FEATURES
# ============================================================================ #

doc_features = np.load(config.BERT_DOC_FEATURES_PATH)
embedding_dim = doc_features.shape[1]
word_features = np.zeros((n_words, embedding_dim), dtype=np.float32)
node_features = np.vstack([doc_features, word_features])

print(f"✓ Node features shape: {node_features.shape}")

# ============================================================================ #
# LABELS AND MASKS
# ============================================================================ #

labels = np.load(config.LABELS_PATH)
train_mask = np.load(config.TRAIN_MASK_PATH)
val_mask = np.load(config.VAL_MASK_PATH)
test_mask = np.load(config.TEST_MASK_PATH)

hetero_labels = np.zeros(n_nodes, dtype=np.int64)
hetero_labels[:n_docs] = labels

hetero_train_mask = np.zeros(n_nodes, dtype=bool)
hetero_val_mask = np.zeros(n_nodes, dtype=bool)
hetero_test_mask = np.zeros(n_nodes, dtype=bool)

hetero_train_mask[:n_docs] = train_mask
hetero_val_mask[:n_docs] = val_mask
hetero_test_mask[:n_docs] = test_mask

print(f"✓ Labels and masks created")

# ============================================================================ #
# SAVE EVERYTHING
# ============================================================================ #

print("\nSaving files...")
sp.save_npz(config.ADJ_OUTPUT_PATH, adj)
np.save(config.NODE_FEATURES_PATH, node_features.astype(np.float32))
np.save(config.LABELS_OUTPUT_PATH, hetero_labels)
np.save(config.TRAIN_MASK_OUTPUT, hetero_train_mask)
np.save(config.VAL_MASK_OUTPUT, hetero_val_mask)
np.save(config.TEST_MASK_OUTPUT, hetero_test_mask)

mapping = {
    'n_docs': n_docs,
    'n_words': n_words,
    'n_nodes': n_nodes,
    'doc_offset': 0,
    'word_offset': n_docs,
    'vocab_size': vocab_size
}

with open(config.NODE_MAPPING_PATH, 'w') as f:
    json.dump(mapping, f, indent=2)

stats = {
    'graph_type': 'Heterogeneous',
    'n_docs': n_docs,
    'n_words': n_words,
    'n_nodes': n_nodes,
    'n_edges': int(adj.nnz),
    'embedding_dim': embedding_dim,
    'window_size': config.WINDOW_SIZE,
    'npmi_threshold': config.NPMI_THRESHOLD,
    'min_pair_count': config.MIN_PAIR_COUNT
}

with open(config.GRAPH_STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)

print("\n" + "="*80)
print("✅ HETEROGENEOUS GRAPH SUCCESSFULLY BUILT!")
print("="*80)
print(f"✓ Adjacency: {adj.shape}")
print(f"✓ Features: {node_features.shape}")
print(f"✓ Total nodes: {n_nodes:,} (docs: {n_docs:,}, words: {n_words:,})")
print(f"✓ Total edges: {adj.nnz:,}")
print("="*80)
print("Ready for training! 🚀")

Recreating doc_word_freq and doc_word_tfidf...
✓ doc_word_freq shape: (122817, 20000)
✓ doc_word_tfidf shape: (122817, 20000)


Rebuilding NPMI edges: 100%|██████████████████████████████████████████| 35719143/35719143 [00:35<00:00, 1012078.40it/s]


✓ Word-word edges rebuilt: 2,476,469
Building adjacency with 35,310,216 edges...
Adding word-word edges...


Word-word edges: 100%|██████████████████████████████████████████████████| 2476469/2476469 [00:02<00:00, 1052031.41it/s]


✓ Added word-word edges, current idx: 4,952,938
Adding doc-word edges...


Doc-word edges: 100%|██████████████████████████████████████████████████| 15178639/15178639 [00:20<00:00, 756792.25it/s]


✓ Total edges added: 35,310,216
Building sparse matrix...
✓ Adjacency shape: (142817, 142817), edges: 35,310,216
✓ Node features shape: (142817, 768)
✓ Labels and masks created

Saving files...

✅ HETEROGENEOUS GRAPH SUCCESSFULLY BUILT!
✓ Adjacency: (142817, 142817)
✓ Features: (142817, 768)
✓ Total nodes: 142,817 (docs: 122,817, words: 20,000)
✓ Total edges: 35,310,216
Ready for training! 🚀
